# Steam Publisher & Developer Analytics
**By Nirmal Kumar Murali**

## What this project does
Analyses 10,000+ Steam games to find which publishers 
have the biggest reach and which make the most 
consistently loved games.

## Key Findings
- Biggest publisher by owners: Valve (150M+ for CS:GO alone)
- Most consistently loved publisher: Fireproof Games (97.5% avg rating)
- Most loved single game: Portal 2 (98.7% positive reviews)

In [19]:
import pandas as pd 
import numpy as np
df = pd.read_csv('steam_games_dataset.csv')
print(df.shape)
df.head()


(10000, 17)


,appid,name,developer,publisher,score_rank,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu
0,730,Counter-Strike: Global Offensive,Valve,Valve,NaN,7642084,1173003,0,"100,000,000 .. 200,000,000",33852,708,6645,301,0,0,0,1013936
1,1172470,Apex Legends,Respawn,Electronic Arts,NaN,668053,326926,0,"100,000,000 .. 200,000,000",10506,496,935,246,0,0,0,124262
2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",NaN,1520457,1037487,0,"100,000,000 .. 200,000,000",23165,717,5622,261,0,0,0,314682
3,1623730,Palworld,Pocketpair,Pocketpair,NaN,358266,22443,0,"50,000,000 .. 100,000,000",3854,835,2213,257,2999,2999,0,18028
4,440,Team Fortress 2,Valve,Valve,NaN,1044264,117208,0,"50,000,000 .. 100,000,000",21244,736,4262,102,0,0,0,43819


In [20]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print()
print(df.dtypes)


Number of rows: 10000
Number of columns: 17

appid                int64
name                   str
developer              str
publisher              str
score_rank         float64
positive             int64
negative             int64
userscore            int64
owners                 str
average_forever      int64
average_2weeks       int64
median_forever       int64
median_2weeks        int64
price                int64
initialprice         int64
discount             int64
ccu                  int64
dtype: object


In [21]:
# Let's first look at what owners actually looks like
print(df['owners'].head(10))

0    100,000,000 .. 200,000,000
1    100,000,000 .. 200,000,000
2    100,000,000 .. 200,000,000
3     50,000,000 .. 100,000,000
4     50,000,000 .. 100,000,000
5     50,000,000 .. 100,000,000
6     50,000,000 .. 100,000,000
7     50,000,000 .. 100,000,000
8     50,000,000 .. 100,000,000
9     50,000,000 .. 100,000,000
Name: owners, dtype: str


In [22]:
#this column owners has unwanted charcters or string but with some pattern 
def parse_owners(text):
    if pd.isnull(text):
        return np.nan
    parts = text.replace(',', '').split('..')
    low = float(parts[0].strip())
    high = float(parts[1].strip())
    return (low + high) / 2

df['owners_mid'] = df['owners'].apply(parse_owners)
    
print(df[['owners', 'owners_mid']].head(10))

                       owners   owners_mid
0  100,000,000 .. 200,000,000  150000000.0
1  100,000,000 .. 200,000,000  150000000.0
2  100,000,000 .. 200,000,000  150000000.0
3   50,000,000 .. 100,000,000   75000000.0
4   50,000,000 .. 100,000,000   75000000.0
5   50,000,000 .. 100,000,000   75000000.0
6   50,000,000 .. 100,000,000   75000000.0
7   50,000,000 .. 100,000,000   75000000.0
8   50,000,000 .. 100,000,000   75000000.0
9   50,000,000 .. 100,000,000   75000000.0


In [23]:
#checking missing values in the whole dataset
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

print(pd.DataFrame({
    'Missing count': missing,
    'Missing percentage' : missing_pct
}
))

                 Missing count  Missing percentage
appid                        0                 0.0
name                         0                 0.0
developer                   33                 0.3
publisher                   54                 0.5
score_rank                9995               100.0
positive                     0                 0.0
negative                     0                 0.0
userscore                    0                 0.0
owners                       0                 0.0
average_forever              0                 0.0
average_2weeks               0                 0.0
median_forever               0                 0.0
median_2weeks                0                 0.0
price                        0                 0.0
initialprice                 0                 0.0
discount                     0                 0.0
ccu                          0                 0.0
owners_mid                   0                 0.0


In [24]:
df = df.drop(columns=['score_rank'])
print("score_rank column removed ✅")

score_rank column removed ✅


In [25]:
df.head()

,appid,name,developer,publisher,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu,owners_mid
0,730,Counter-Strike: Global Offensive,Valve,Valve,7642084,1173003,0,"100,000,000 .. 200,000,000",33852,708,6645,301,0,0,0,1013936,150000000.0
1,1172470,Apex Legends,Respawn,Electronic Arts,668053,326926,0,"100,000,000 .. 200,000,000",10506,496,935,246,0,0,0,124262,150000000.0
2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",1520457,1037487,0,"100,000,000 .. 200,000,000",23165,717,5622,261,0,0,0,314682,150000000.0
3,1623730,Palworld,Pocketpair,Pocketpair,358266,22443,0,"50,000,000 .. 100,000,000",3854,835,2213,257,2999,2999,0,18028,75000000.0
4,440,Team Fortress 2,Valve,Valve,1044264,117208,0,"50,000,000 .. 100,000,000",21244,736,4262,102,0,0,0,43819,75000000.0


In [26]:
# total reviews per game
df['total_reviews'] = df['positive'] + df['negative']

# % of  positive?
df['positive_ratio'] = (df['positive'] / df['total_reviews'] * 100).round(1)
#The current price of the game is in cents
# price from cents to euros
df['price_eur'] = (df['price'] / 100).round(2)

# playtime from minutes to hours
df['avg_playtime_hrs'] = (df['average_forever'] / 60).round(1)

print("New columns added:")
print(df[['name', 'total_reviews', 'positive_ratio', 'price_eur', 'avg_playtime_hrs']].head())

New columns added:
                               name  total_reviews  positive_ratio  price_eur  \
0  Counter-Strike: Global Offensive        8815087            86.7       0.00   
1                      Apex Legends         994979            67.1       0.00   
2               PUBG: BATTLEGROUNDS        2557944            59.4       0.00   
3                          Palworld         380709            94.1      29.99   
4                   Team Fortress 2        1161472            89.9       0.00   

   avg_playtime_hrs  
0             564.2  
1             175.1  
2             386.1  
3              64.2  
4             354.1  


In [27]:
#  games  less than 100 reviews

df_filtered = df[df['total_reviews'] >= 100].copy()

print(f"Original dataset: {len(df)} games")
print(f"After filtering:  {len(df_filtered)} games")
print(f"Removed:          {len(df) - len(df_filtered)} games with less than 100 reviews")

Original dataset: 10000 games
After filtering:  9108 games
Removed:          892 games with less than 100 reviews


In [28]:
# Publisher Analysis
# Group all games by publisher and calculate portfolio stats

publisher_stats = df_filtered.groupby('publisher').agg(
    game_count    = ('name', 'count'),
    total_owners  = ('owners_mid', 'sum'),
    avg_rating    = ('positive_ratio', 'mean'),
    total_reviews = ('total_reviews', 'sum'),
    avg_price     = ('price_eur', 'mean'),
    peak_ccu      = ('ccu', 'max')
).reset_index()

# Round the decimals
publisher_stats['avg_rating'] = publisher_stats['avg_rating'].round(1)
publisher_stats['avg_price']  = publisher_stats['avg_price'].round(2)

# Sort by total owners - biggest publishers first
publisher_stats = publisher_stats.sort_values('total_owners', ascending=False)

print(publisher_stats.head(10))

               publisher  game_count  total_owners  avg_rating  total_reviews  \
3819               Valve          33   568200000.0        90.0       14138556   
1144     Electronic Arts         110   364500000.0        78.4        4818220   
3757             Ubisoft         117   205075000.0        77.4        4073088   
3033      Rockstar Games          23   171975000.0        82.6        3526282   
1921       KRAFTON, Inc.           8   158400000.0        77.4        2705196   
219         Amazon Games           3   153500000.0        60.9         499643   
145           Activision          46   151300000.0        81.8        1463546   
450   Bethesda Softworks          56   148000000.0        81.8        3242623   
606     CAPCOM Co., Ltd.          38   122325000.0        82.8        2060809   
4001   Xbox Game Studios          52   117350000.0        84.6        2597871   

      avg_price  peak_ccu  
3819       9.39   1013936  
1144      25.24    124262  
3757       6.58     5184

In [29]:
# Valve's games stats, since it came no.1 in the list
valve_games = df_filtered[df_filtered['publisher'] == 'Valve']

print(valve_games[['name', 'owners_mid', 'positive_ratio', 'avg_playtime_hrs']].to_string())

                                  name   owners_mid  positive_ratio  avg_playtime_hrs
0     Counter-Strike: Global Offensive  150000000.0            86.7             564.2
4                      Team Fortress 2   75000000.0            89.9             354.1
9                        Left 4 Dead 2   75000000.0            97.5              41.3
22                         Garry's Mod   35000000.0            96.8             158.4
36             Half-Life 2: Lost Coast   35000000.0            89.3               0.0
50             Half-Life 2: Deathmatch   15000000.0            90.7              16.1
51                           Half-Life   15000000.0            96.5              10.0
63                      Counter-Strike   15000000.0            97.4             191.3
64              Counter-Strike: Source   15000000.0            96.3             104.7
67                              Portal   15000000.0            98.5               6.6
74      Counter-Strike: Condition Zero   15000000.0   

In [30]:
# "Which Valve game has the highest positive ratio?"
valve_game_stats = valve_games.sort_values('positive_ratio' , ascending=False)
print(valve_game_stats[['name', 'positive_ratio']].head(5))


                name  positive_ratio
172         Portal 2            98.7
67            Portal            98.5
650  Half-Life: Alyx            98.3
81       Half-Life 2            97.6
9      Left 4 Dead 2            97.5


In [31]:
# Most loved publishers (minimum 5 games)
most_loved = publisher_stats[publisher_stats['game_count'] >= 5]. sort_values(
    'avg_rating', ascending= False
)
print(most_loved[['publisher', 'game_count', 'avg_rating', 'total_owners']].head(10).to_string())

                                publisher  game_count  avg_rating  total_owners
1272                      Fireproof Games           5        97.5     2350000.0
3062                           Rusty Lake           8        96.8     5150000.0
2817  PopCap Games, Inc., Electronic Arts          10        96.0     7575000.0
1594                         HIKARI FIELD           7        95.9      975000.0
3464                     Supergiant Games           5        94.6    16750000.0
2085                     Lil Hentai Games           5        94.0      875000.0
2881                          Quiet River          16        93.9     4200000.0
2456                New Blood Interactive           7        93.7    12025000.0
1054                              DreadXP           8        93.0     1800000.0
3156                        Scott Cawthon           8        92.7    15000000.0


In [33]:
# Export clean data for Power BI and future use
df_filtered.to_csv('steam_cleaned.csv', index=False)
publisher_stats.to_csv('publisher_stats.csv', index=False)

print("✅ Files exported successfully!")
print(f"   steam_cleaned.csv     → {len(df_filtered)} rows")
print(f"   publisher_stats.csv   → {len(publisher_stats)} rows")

✅ Files exported successfully!
   steam_cleaned.csv     → 9108 rows
   publisher_stats.csv   → 4289 rows
